### Word Embedding Analysis & Vector Math using GloVe

This notebook demonstrates key vector space operations and analysis techniques using GloVe embeddings. This uses glove-wiki-gigaword-100 (100 dimensions, ~128 MB) via gensim.downloader.

***Prerequisites***
Install the required packages before running:
```bash
pip install gensim numpy pandas matplotlib seaborn

In [ ]:
import gensim.downloader as api
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

print("Loading lightweight GloVe model ('glove-wiki-gigaword-100')...")
glove = api.load("glove-wiki-gigaword-100")
print(f"Successfully loaded {len(glove.index_to_key):,} words with {glove.vector_size}D dimensions!\n")

Vector Mathematics & Search Helper Functions: Helper functions for cosine similarity, nearest neighbors, semantic outlier detection, and vector analogy solving using Gensim matrix operations.

In [ ]:
def get_cosine_similarity(v1: np.ndarray, v2: np.ndarray) -> float:
    norm1 = np.linalg.norm(v1)
    norm2 = np.linalg.norm(v2)
    if norm1 == 0 or norm2 == 0:
        return 0.0
    return float(np.dot(v1, v2) / (norm1 * norm2))

def get_nearest_neighbors(model, word: str, topn: int = 10):
    word = word.lower()
    if word not in model:
        print(f"Word '{word}' not found in vocabulary.")
        return []
    return model.most_similar(word, topn=topn)

def get_doesnt_match(model, words: list) -> str:
    valid_words = [w.lower() for w in words if w.lower() in model]
    if len(valid_words) < 2:
        return words[0]
    return model.doesnt_match(valid_words)

def solve_analogy(model, w1: str, w2: str, w3: str, topn: int = 1):
    w1, w2, w3 = w1.lower(), w2.lower(), w3.lower()
    if not all(w in model for w in [w1, w2, w3]):
        return []
    return model.most_similar(positive=[w1, w3], negative=[w2], topn=topn)

Global Vocabulary Statistics & Word Length Distribution: Explores word length distributions and inspects token index ordering across the vocabulary.

In [ ]:
vocab_size = len(glove.index_to_key)
print(f"Total vocabulary size : {vocab_size:,} tokens")
print(f"Embedding dimensions  : {glove.vector_size}D")

sample_indices = [10, 500, 12000, 45000, 180000]
for idx in sample_indices:
    if idx < vocab_size:
        print(f"  Index {idx:6d}: {glove.index_to_key[idx]}")

word_lengths = [len(word) for word in glove.index_to_key]
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(word_lengths, bins=range(1, 35), color="skyblue", edgecolor="black")
axes[0].set_title("Word Length Distribution (Linear Scale)")
axes[1].hist(word_lengths, bins=range(1, 35), color="skyblue", edgecolor="black", log=True)
axes[1].set_title("Word Length Distribution (Log Scale)")
plt.tight_layout()
plt.show()

Global Vector Space Statistics: Calculates mean and standard deviation across embedding vectors and visualizes their overall distribution using Seaborn.

In [ ]:
vectors = glove.vectors
means = np.mean(vectors, axis=1)
stds = np.std(vectors, axis=1)

df = pd.DataFrame({"Mean": means, "Standard Deviation": stds})
grid = sns.jointplot(data=df, x="Mean", y="Standard Deviation", alpha=0.1, color="teal", height=6)
grid.fig.suptitle("Mean vs Std Dev Across Vocabulary Vectors", y=1.02)
plt.show()

Dimension Profiles & Cosine Similarity Scatter Plots: Compares dimension profiles and pairwise spatial alignment between related and unrelated terms (`banana`, `apple`, `cosmic`).

In [ ]:
v_banana, v_apple, v_cosmic = glove["banana"], glove["apple"], glove["cosmic"]

plt.figure(figsize=(10, 4))
plt.plot(v_banana, label="banana", color="blue")
plt.plot(v_apple, label="apple", color="orange")
plt.plot(v_cosmic, label="cosmic", color="green")
plt.title("Dimension Profiles for 'banana', 'apple', and 'cosmic'")
plt.legend()
plt.grid(True, linestyle="--", alpha=0.5)
plt.show()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
ax1.scatter(v_banana, v_apple, color="purple", alpha=0.5)
ax1.set_title(f"Banana vs Apple (Cos Sim: {get_cosine_similarity(v_banana, v_apple):.4f})")
ax2.scatter(v_banana, v_cosmic, color="darkgreen", alpha=0.5)
ax2.set_title(f"Banana vs Cosmic (Cos Sim: {get_cosine_similarity(v_banana, v_cosmic):.4f})")
plt.tight_layout()
plt.show()

Nearest Neighbors & Outlier Detection: Evaluates taxonomy neighbor discovery and identifies out-of-context terms in word groups.

In [ ]:
for target in ["frog", "fashion"]:
    print(f"Closest words to '{target}':", get_nearest_neighbors(glove, target, topn=5))

print("Outlier in ['apple', 'banana', 'mango', 'pirate'] ->", get_doesnt_match(glove, ["apple", "banana", "mango", "pirate"]))

Linear Substructures & Vector Analogies: Demonstrates algebraic vector analogies (e.g. $v_{\text{king}} - v_{\text{man}} + v_{\text{woman}} \approx v_{\text{queen}}$) and vector offset alignments.

In [ ]:
analogies = [
    ("king", "man", "woman", "King - Man + Woman -> Queen"),
    ("paris", "france", "germany", "Paris - France + Germany -> Berlin"),
    ("slower", "slow", "fast", "Slower - Slow + Fast -> Faster"),
]

for w1, w2, w3, desc in analogies:
    results = solve_analogy(glove, w1, w2, w3, topn=1)
    top_word, score = results[0] if results else ("N/A", 0.0)
    print(f"Analogy: {desc:38s} => Result: {top_word} ({score:.4f})")

v_man_woman = glove["man"] - glove["woman"]
v_king_queen = glove["king"] - glove["queen"]
v_brother_sister = glove["brother"] - glove["sister"]

sim_1 = get_cosine_similarity(v_man_woman, v_king_queen)
sim_2 = get_cosine_similarity(v_man_woman, v_brother_sister)

print("\nVector Difference Substructure Alignment:")
print(f"  Cosine Sim (Man - Woman, King - Queen)     : {sim_1:.4f}")
print(f"  Cosine Sim (Man - Woman, Brother - Sister) : {sim_2:.4f}\n")

Co-occurrence Probability Ratios: Examines how GloVe models log co-occurrence probability ratios through vector dot products.

In [ ]:
target_ice = glove["ice"]
target_steam = glove["steam"]
probe_words = ["solid", "gas", "water", "fashion"]

print(f"{'Probe Word (x)':15s} | {'dot(v_ice, x)':15s} | {'dot(v_steam, x)':15s} | {'Implied Ratio [P(x|ice)/P(x|steam)]':35s}")
print("-" * 85)

for word in probe_words:
    if word in glove:
        dot_ice = np.dot(target_ice, glove[word])
        dot_steam = np.dot(target_steam, glove[word])
        implied_ratio = np.exp(dot_ice - dot_steam)
        print(f"{word:15s} | {dot_ice:15.4f} | {dot_steam:15.4f} | {implied_ratio:35.4f}")

Heatmap & Density Banding Visualizations: Visualizes horizontal banding across related conceptual groups and vertical density trends across frequency indices.

In [ ]:
words_to_plot = ["water", "ice", "steam", "king", "queen", "man", "woman"]
matrix = np.array([glove[w] for w in words_to_plot if w in glove])

plt.figure(figsize=(12, 4))
sns.heatmap(matrix, yticklabels=words_to_plot, cmap="viridis")
plt.title("GloVe Heatmap Across Dimensions")
plt.show()